# Tabelle zusammenlegen

In [55]:
import pandas as pd

# 1. Bereinigte Daten laden
df_orders = pd.read_csv("Daten/clean/orders_clean.csv")
df_olines = pd.read_csv("Daten/clean/orderlines_clean.csv")
df_products = pd.read_csv("Daten/clean/products_clean.csv")


# 2. Zusammenlegen (Merge) mittels order_id und id_order
# Wir nutzen einen "inner" Join, um nur die Zeilen zu behalten, die in beiden Tabellen existieren.
# Falls du auch Bestellungen ohne Orderlines behalten willst, ändere 'inner' zu 'left'.
df_merged = pd.merge(df_orders, df_olines, left_on='order_id', right_on='id_order', how='inner')
df_order_sort = ["order_id", "sku", "product_quantity", "unit_price", "total_paid"]
df_merged = df_merged[df_order_sort]

display(df_merged.head(100))

df_merged.to_csv("Daten/clean/order_sort.csv", index=False)

,order_id,sku,product_quantity,unit_price,total_paid
0,241319,JBL0123,1,44.99,44.99
1,241423,LAC0212,1,129.16,136.15
2,242832,PAR0074,1,10.77,15.76
3,243330,OWC0074,1,77.99,84.98
4,243784,PHI0080,3,51.29,157.86
...,...,...,...,...,...
95,268362,CRU0055,1,469.90,573.21
96,268362,SEA0053,1,103.31,573.21
97,268372,FCM0007-2,1,112.09,1243.53
98,268659,APP0608,1,279.99,279.99


## Versandkosten ermitteln

In [56]:
# 1. Wir berechnen den echten Warenwert pro Zeile (Menge * Preis)
df_merged['line_total'] = df_merged['product_quantity'] * df_merged['unit_price']

# 2. Wir summieren den gesamten Warenwert für JEDE Order (Hilfsspalte im Hintergrund)
df_merged['order_totals'] = df_merged.groupby('order_id')['line_total'].transform('sum')
# Jetzt zwingen wir diese offizielle Spalte auf 2 Kommastellen
df_merged['order_totals'] = df_merged['order_totals'].round(2)
# 3. Versandkosten ermitteln (Bezahlt - Warenwert)
df_merged['shipping_cost'] = (df_merged['total_paid'] - df_merged['order_totals']).round(2)

df_merged.to_csv("Daten/clean/versand.csv", index=False)

# 4. Das "Total-Paid-Problem" lösen! 
# Wir suchen uns alle Zeilen, bei denen die Order-ID schon zum zweiten, dritten, ... mal auftaucht
is_duplicate = df_merged.duplicated(subset=['order_id'])

# Bei all diesen Folge-Zeilen setzen wir total_paid UND die Versandkosten auf 0!
# So stehen diese Beträge immer nur exakt EINMAL pro Bestellung (in der ersten Zeile).
df_merged.loc[is_duplicate, 'total_paid'] = None
df_merged.loc[is_duplicate, 'shipping_cost'] = None



# 5. Kontrolle
# Lass uns wieder deine problematische Order 246018 anzeigen, um zu sehen, ob es geklappt hat:
display(df_merged[df_merged['order_id'] == 246018])


,order_id,sku,product_quantity,unit_price,total_paid,line_total,order_totals,shipping_cost
8,246018,IFX0055,1,93.99,211.95,93.99,206.96,4.99
9,246018,TUC0308,1,24.99,NaN,24.99,206.96,NaN
10,246018,IFX0020,1,7.99,NaN,7.99,206.96,NaN
11,246018,MOP0089,1,79.99,NaN,79.99,206.96,NaN


### Versandkosten anzeigen

In [57]:
# 1. Wir filtern das DataFrame, sodass wir nur die jeweils erste Zeile einer Order anschauen
echte_bestellungen = df_merged[~df_merged.duplicated(subset=['order_id'])]

# 2. Wir runden sicherheitshalber auf 2 Kommastellen (vermeidet Fehler wie 6.99000001)
versand_gerundet = echte_bestellungen['shipping_cost'].round(2)

# 3. Zählen, wie oft welcher Versandbetrag vorkommt
# sort_index() sortiert die Ausgabe vom niedrigsten bis zum höchsten Betrag
versand_counts = versand_gerundet.value_counts().sort_index()
#######
print("--- DIE 10 NIEDRIGSTEN VERSANDKOSTEN (Achtung vor Negativwerten!) ---")
print(versand_counts.head(10))

print("\n--- DIE 10 HÄUFIGSTEN VERSANDKOSTEN (Das ist der Standard) ---")
print(versand_gerundet.value_counts().head(200))

print("\n--- DIE 10 HÖCHSTEN VERSANDKOSTEN (Hier verstecken sich Ausreißer!) ---")
print(versand_counts.tail(100))


--- DIE 10 NIEDRIGSTEN VERSANDKOSTEN (Achtung vor Negativwerten!) ---
shipping_cost
-381.47    1
-362.00    1
-359.99    1
-333.00    1
-236.41    1
-215.61    1
-165.00    1
-150.58    1
-147.19    1
-120.02    1
Name: count, dtype: int64

--- DIE 10 HÄUFIGSTEN VERSANDKOSTEN (Das ist der Standard) ---
shipping_cost
0.00       120551
4.99        15856
6.99        14989
3.99        10285
0.01         2639
            ...  
2498.59        20
2709.59        20
1263.00        20
1585.60        20
2643.59        20
Name: count, Length: 200, dtype: int64

--- DIE 10 HÖCHSTEN VERSANDKOSTEN (Hier verstecken sich Ausreißer!) ---
shipping_cost
9065.96      2
9218.38      1
9269.98      1
9286.38      1
9411.96      1
            ..
189040.05    1
194640.82    1
208847.23    1
209756.47    1
211728.78    1
Name: count, Length: 100, dtype: int64


In [58]:
# 1. Zuerst speichern wir den Gesamtwarenwert pro Bestellung dauerhaft als Spalte ab
df_merged['order_totals'] = df_merged.groupby('order_id')['line_total'].transform('sum')

# 2. Wir filtern auf ECHTE Bestellungen (nur die erste Zeile pro order_id)
echte_bestellungen = df_merged[~df_merged.duplicated(subset=['order_id'])].copy()

# 3. Wir runden die Versandkosten, um saubere Cluster zu bekommen
echte_bestellungen['shipping_cost'] = echte_bestellungen['shipping_cost'].round(2)

print("--- 1. CLUSTER-ANALYSE: DIE 10 HÄUFIGSTEN VERSANDKOSTEN ---")
versand_counts = echte_bestellungen['shipping_cost'].value_counts()
print(versand_counts.head(10))

print("\n--------------------------------------------------------------")

print("\n--- 2. SCHWELLENWERT-ANALYSE FÜR GRATISVERSAND (0.00 €) ---")
# Filtern auf alle Bestellungen, die exakt 0.00 € Versand gekostet haben
gratis_versand = echte_bestellungen[echte_bestellungen['shipping_cost'] == 0.0]

print(f"Anzahl Bestellungen mit Gratisversand: {len(gratis_versand)}")

if len(gratis_versand) > 0:
    print(f"Geringster Warenwert, der Gratisversand bekam: {gratis_versand['order_totals'].min()} €")
    print(f"Durchschnittlicher Warenwert bei Gratisversand: {gratis_versand['order_totals'].mean():.2f} €")
    
    print("\nHier sind die 15 kleinsten Warenwerte mit Gratisversand:")
    # Das hilft dir zu sehen, ob z.B. alles ab 50€ gratis ist, oder ob es krasse Ausnahmen gibt
    print(gratis_versand['order_totals'].sort_values().head(15).values)


--- 1. CLUSTER-ANALYSE: DIE 10 HÄUFIGSTEN VERSANDKOSTEN ---
shipping_cost
 0.00     120551
 4.99      15856
 6.99      14989
 3.99      10285
 0.01       2639
 19.99      1233
 9.99        936
-0.01        907
 5.00        397
 7.00        347
Name: count, dtype: int64

--------------------------------------------------------------

--- 2. SCHWELLENWERT-ANALYSE FÜR GRATISVERSAND (0.00 €) ---
Anzahl Bestellungen mit Gratisversand: 120551
Geringster Warenwert, der Gratisversand bekam: 0.0 €
Durchschnittlicher Warenwert bei Gratisversand: 244.22 €

Hier sind die 15 kleinsten Warenwerte mit Gratisversand:
[0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]


In [59]:
# Wir filtern die echten Gratisversände, ABER der Warenwert muss größer als 0 Euro sein!
echter_gratis_versand = gratis_versand[gratis_versand['order_totals'] > 0.0]

print("Hier sind die kleinsten ECHTEN Warenwerte mit Gratisversand:")
print(echter_gratis_versand['order_totals'].sort_values().head(20).values)


Hier sind die kleinsten ECHTEN Warenwerte mit Gratisversand:
[0.29 1.33 1.33 1.33 1.33 1.99 1.99 2.31 2.49 2.49 2.49 2.49 2.49 2.49
 2.49 2.49 2.49 2.49 2.49 2.54]


## IQR-Code für die Versandkosten

1. Wir betrachten für die Berechnung der Grenzen nur EINE Zeile pro Bestellung!
2. Quartile (Q1 & Q3) und IQR berechnen 
3. Die Grenzwerte definieren
4. Liste aller 'guten' Order-IDs erstellen, die INNERHALB der Grenzen liegen, Wir extrahieren nur die Liste der sauberen IDs
5. Datensatz bereinigen: Wir behalten in unserer Master-Tabelle NUR Zeilen, deren order_id in der "sauberen" Liste steht

In [60]:
echte_bestellungen = df_merged[~df_merged.duplicated(subset=['order_id'])]


q1 = echte_bestellungen['shipping_cost'].quantile(0.25)
q3 = echte_bestellungen['shipping_cost'].quantile(0.75)
iqr = q3 - q1

lower_bound = q1 - 1.5 * iqr
upper_bound = q3 + 1.5 * iqr
print(f"Statistische Grenzen für korrekte Versandkosten: {lower_bound} € bis {upper_bound} €")


gute_bestellungen = echte_bestellungen[
    (echte_bestellungen['shipping_cost'] >= lower_bound) & 
    (echte_bestellungen['shipping_cost'] <= upper_bound)
]


saubere_order_ids = gute_bestellungen['order_id']
df_clean = df_merged[df_merged['order_id'].isin(saubere_order_ids)]

print(f"\nUrsprüngliche Zeilen: {len(df_merged)}")
print(f"Zeilen nach IQR-Bereinigung: {len(df_clean)}")


Statistische Grenzen für korrekte Versandkosten: -7.485 € bis 12.475000000000001 €

Ursprüngliche Zeilen: 257611
Zeilen nach IQR-Bereinigung: 213825


## Products_clean zusammenführen
sku, name

In [61]:
df_clean.tail(10)
#df_merged = pd.merge(df_merged, df_products, on='sku', how='left')
#df_clean = pd.merge(df_olines, df_products[['sku', 'name']], on='sku', how='left')
df_clean.head()

,order_id,sku,product_quantity,unit_price,total_paid,line_total,order_totals,shipping_cost
0,241319,JBL0123,1,44.99,44.99,44.99,44.99,0.00
1,241423,LAC0212,1,129.16,136.15,129.16,129.16,6.99
2,242832,PAR0074,1,10.77,15.76,10.77,10.77,4.99
3,243330,OWC0074,1,77.99,84.98,77.99,77.99,6.99
4,243784,PHI0080,3,51.29,157.86,153.87,153.87,3.99


### Komplett speichern

In [62]:


df_clean.to_csv("Daten/clean/order_sort.csv", index=False, float_format='%.2f')

### In Teilen speichern

In [63]:
df_clean.head(29).to_csv("Daten/clean/order_sort_small.csv", index=False, float_format="%.2f")